# Backtest MACrossBracket

Run the MA regime-based bracket strategy on a simulated exchange venue.
Supports all MA types: EMA, SMA, HMA, DEMA, AMA (Kaufman), VIDYA.

**Strategy logic:**
- **Entry:** MA regime-based (enters every bar while flat and fast MA >= slow for longs, fast < slow for shorts)
- **Exit:** Symmetric ATR-sized bracket orders — market entry + limit TP + stop-market SL at the same distance
- NT cancels the remaining bracket leg automatically when either TP or SL fills
- After TP/SL fill, re-enters on the next bar if still in MA regime (multiple trades per regime)
- MA reversal while in position: cancel bracket, close via market, enter opposite direction immediately

**Key difference from MACrossATR:**
MACrossATR enters only on crossover bars (one trade per regime, asymmetric R:R).
MACrossBracket re-enters after every TP/SL fill while still in regime (multiple trades per regime, symmetric 1:1 R:R).

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + MA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweeps — MA period grid, then bracket distance × ATR period sensitivity

**Prerequisites:** Run `scripts/fetch_binance_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_sweep
from src.core import bar_type_str, with_venue_config
from src.strategies.ma_cross_bracket import MACrossBracket, MACrossBracketConfig

from charts import plot_ma_cross, plot_equity_curve, print_summary_stats, plot_pnl_heatmap, generate_backtest_html
from utils import make_instrument_id, save_notebook, save_notebook_html, save_tearsheet

# ── Shared config ────────────────────────────────────────────────────
EXCHANGE         = "BINANCE"
ASSET            = "BTC"
BAR_INTERVAL     = "4h"
STARTING_CAPITAL = 10_000
TRADE_SIZE       = 1000              # $1,000 USD per trade
LEVERAGE         = 20                # Backtesting leverage (margin_init = 1/20 = 5%)
SAVE_TEARSHEET   = False

# MA params
MA_TYPE = "EMA"
FAST_MA = 10
SLOW_MA = 20

# ATR bracket params
ATR_PERIOD    = 20
BRACKET_DIST  = 3.0    # Symmetric: SL and TP both at ATR × 3.0

# Sweep grids — MA periods
FAST_PERIODS = sorted(set([5, 8, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50] + [FAST_MA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_MA]))

# Sweep grids — bracket distance and ATR period
BRACKET_DISTS = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
ATR_PERIODS   = [10, 14, 20, 30]

# Standard
TRADE_NOTIONAL = Decimal(TRADE_SIZE)
INSTRUMENT_ID  = make_instrument_id(ASSET, EXCHANGE)
TRADING_PAIR   = INSTRUMENT_ID.split("-")[0]
BAR_TYPE_STR   = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE          = Venue(EXCHANGE)

# Files
CATALOG_PATH = "../data/catalog"
RESULT_NAME  = f"MACrossBracket-{MA_TYPE}_{INSTRUMENT_ID}_{BAR_INTERVAL}_f{FAST_MA}_s{SLOW_MA}_dist{BRACKET_DIST}"

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

# Override margins for backtesting (catalog instruments have raw/live margins)
instrument = with_venue_config(instrument, LEVERAGE)

# Settlement currency for PnL queries (USDC for HL, USDT for Binance)
CURRENCY = instrument.settlement_currency

print(f"Instrument : {instrument.id}")
print(f"Currency   : {CURRENCY}")
print(f"Leverage   : {LEVERAGE}x (margin_init={instrument.margin_init}, margin_maint={instrument.margin_maint})")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add MACrossBracket strategy + run ────────────────
config = MACrossBracketConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_notional=TRADE_NOTIONAL,
    ma_type=MA_TYPE,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    atr_period=ATR_PERIOD,
    bracket_distance_atr=BRACKET_DIST,
)
strategy = MACrossBracket(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ────────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ──────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"MACrossBracket-{MA_TYPE}({FAST_MA}/{SLOW_MA}, dist={BRACKET_DIST}x) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

if SAVE_TEARSHEET:
    save_tearsheet(html, RESULT_NAME)

In [ ]:
# ── Cell 8: Price chart with MA overlay + trade markers ────────────
#
# Reuses plot_ma_cross() — same MA indicators, trade markers show
# bracket entries, TP fills, and SL fills.

fig = plot_ma_cross(
    bars, fills_report,
    ma_type=MA_TYPE,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ────────────────────────────────────────────
plot_equity_curve(
    analyzer, account_report,
    f"MACrossBracket-{MA_TYPE}({FAST_MA}/{SLOW_MA}, ATR={ATR_PERIOD}, dist={BRACKET_DIST}×)  {INSTRUMENT_ID} {BAR_INTERVAL}",
)

In [ ]:
# ── Cell 10: Summary statistics ─────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions), currency=CURRENCY)

In [ ]:
# ── Cell 11: Parameter sweep — MA fast/slow periods ─────────────
#
# Grid search over MA period combinations.
# ATR and bracket distance held at defaults.

def ma_bracket_factory(eng, params):
    cfg = MACrossBracketConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_notional=TRADE_NOTIONAL,
        ma_type=MA_TYPE,
        fast_period=params.get("fast", FAST_MA),
        slow_period=params.get("slow", SLOW_MA),
        atr_period=params.get("atr_period", ATR_PERIOD),
        bracket_distance_atr=params.get("bracket_dist", BRACKET_DIST),
    )
    eng.add_strategy(MACrossBracket(cfg))

combos = [{"fast": f, "slow": s} for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]

results_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ma_bracket_factory,
    strategy_name=f"MACrossBracket-{MA_TYPE}",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

In [ ]:
# ── Cell 12: PnL heatmap — MA fast vs slow ────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label=f"Slow {MA_TYPE} Period", col_label=f"Fast {MA_TYPE} Period",
    title=f"Total PnL ({CURRENCY}) by {MA_TYPE} Parameters",
)

In [ ]:
# ── Cell 13: Bracket distance × ATR period sensitivity sweep ───────
#
# Fix MA at the best combo from the previous sweep.
# Vary bracket distance and ATR period together — tests whether the
# optimal distance depends on the ATR measurement window.

best_row = results_df.loc[results_df["total_pnl"].idxmax()]
best_fast = int(best_row["fast"])
best_slow = int(best_row["slow"])
print(f"Best {MA_TYPE} params: fast={best_fast}, slow={best_slow} (PnL={best_row['total_pnl']:,.2f})")

combos = [{"bracket_dist": d, "atr_period": ap, "fast": best_fast, "slow": best_slow}
          for d in BRACKET_DISTS for ap in ATR_PERIODS]

dist_df = run_sweep(
    venue=VENUE, instrument=instrument, bars=bars,
    starting_capital=STARTING_CAPITAL,
    param_combos=combos,
    strategy_factory=ma_bracket_factory,
    strategy_name=f"MACrossBracket-{MA_TYPE}-dist-sensitivity",
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
)

plot_pnl_heatmap(
    dist_df, row_col="atr_period", col_col="bracket_dist",
    row_label="ATR Period", col_label="Bracket Distance (×ATR)",
    title=f"Total PnL ({CURRENCY}) by Bracket Distance vs ATR Period — {MA_TYPE}({best_fast}/{best_slow})",
)

In [ ]:
# ── Cell 14: TradingView Interactive Backtest ────────────────────

path = generate_backtest_html(
    bars, fills_report, positions_report,
    fast_period=FAST_MA,
    slow_period=SLOW_MA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
    starting_capital=STARTING_CAPITAL,
    result_filename=RESULT_NAME,
)
# Then just open the file in your browser

In [ ]:
# ── Cell 15: Save notebook snapshot ────────────────────────────────────
#
# Save the notebook (Ctrl+S) before running this cell.

#save_notebook("backtest_ema_cross_bracket.ipynb", RESULT_NAME)
#save_notebook_html("backtest_ema_cross_bracket.ipynb", RESULT_NAME)

In [ ]:
# ── Cell 16: Cleanup ──────────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")